
# **IMDB Movie Reviews Dataset: Preprocessing**

The IMDB Movie Reviews Dataset is a widely used benchmark dataset for **sentiment analysis** in natural language processing. It contains a large collection of movie reviews from the Internet Movie Database (IMDb), where each review is labeled according to the sentiment expressed by the author.

The dataset contains two main columns:

| Column    | Description                             |
| --------- | --------------------------------------- |
| **text**  | The full movie review written by a user |
| **label** | The sentiment label of the review       |

The sentiment labels are encoded as:

| Label | Meaning         |
| ----- | --------------- |
| **0** | Negative review |
| **1** | Positive review |

Each row in the dataset represents one movie review and its corresponding sentiment label.



In [47]:
import pandas as pd 
import numpy as np
import matplotlib as mpl
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from datasets import load_dataset
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
import os

# **1: Load the dataset**

In [48]:
#import sys
#!{sys.executable} -m pip install nltk

In [49]:
imdb = load_dataset("imdb")
df_imdb = imdb['train'].to_pandas()
df_imdb.head()

,text,label
0,I rented I AM CURIOUS-YELLOW from my video sto...,0
1,"""I Am Curious: Yellow"" is a risible and preten...",0
2,If only to avoid making this type of film in t...,0
3,This film was probably inspired by Godard's Ma...,0
4,"Oh, brother...after hearing about this ridicul...",0


# **2: Preprocessing**

In [50]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ale\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [51]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

------

# **Pipeline 1: Classical NLP Preprocessing**

This preprocessing pipeline is designed as the first experimental preprocessing
strategy for the IMDB sentiment analysis project.

The objective of this pipeline is to transform raw textual movie reviews into a
cleaner and more standardized representation suitable for classical NLP
architectures.

The pipeline performs the following operations sequentially:

1. Remove special characters and numbers
2. Tokenize the text into words
3. Convert all tokens to lowercase
4. Remove stopwords
5. Apply lemmatization
6. Reconstruct the cleaned text

This pipeline prioritizes:
- Simplicity
- Interpretability
- Compatibility with multiple NLP architectures
- Reduced vocabulary dimensionality

After preprocessing, the dataset is divided into:
- Training set
- Validation set
- Test set

The processed datasets are stored inside a directory called:

```text
preprocessed_data/
```

This allows the same preprocessing pipeline to be reused later for:
- TF-IDF experiments
- Embedding architectures
- Classification models
- Comparative NLP experiments


### **Pipeline 1 Implementation**

In [52]:
import os
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from nltk.tokenize import word_tokenize

# -----------------------------------------
# Pipeline 1: Classical NLP Preprocessing
# -----------------------------------------

def preprocessing_pipeline_1(text):

    """
    Pipeline 1 for IMDB preprocessing.

    Steps:
    1. Remove special characters
    2. Tokenization
    3. Lowercasing
    4. Stopword removal
    5. Lemmatization
    6. Join cleaned tokens

    Returns:
        Cleaned text as a string.
    """

    # Remove special characters and numbers
    # Keeps only alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))

    # Tokenization
    tokens = word_tokenize(text)

    # Stores processed words
    processed_tokens = []

    # Token processing loop
    for word in tokens:

        # Lowercase conversion and whitespace cleaning
        word = word.lower().strip()
        # Filtering conditions:
        # - Remove stopwords
        # - Remove short tokens
        # - Keep only alphabetic tokens
        if (
            word not in stop_words
            and len(word) > 1
            and word.isalpha()
        ):

            # Lemmatization
            word = lemmatizer.lemmatize(word)

            # Add cleaned token
            processed_tokens.append(word)

    # Reconstruct cleaned text
    cleaned_text = " ".join(processed_tokens)

    return cleaned_text


# Complete Pipeline Function
def pipeline_1(df, output_dir="preprocessed_data_pipeline_1"):

    """
    Applies preprocessing pipeline 1,
    performs train/validation/test split,
    and saves processed datasets.
    """
    print("Starting Pipeline 1 Preprocessing...")
    df = df.copy()
    df['clean_text'] = df['text'].apply(preprocessing_pipeline_1)
    df_clean = df[['clean_text', 'label']]

    # Train / Validation / Test Split
    # 80% train
    # 20% temporary
    X_train, X_temp, y_train, y_temp = train_test_split(
        df_clean['clean_text'],
        df_clean['label'],
        test_size=0.2,
        random_state=42,
        stratify=df_clean['label']
    )

    # 10% validation
    # 10% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    # Create DataFrames
    train_df = pd.DataFrame({
        'text': X_train.values,
        'label': y_train.values
    })

    val_df = pd.DataFrame({
        'text': X_val.values,
        'label': y_val.values
    })

    test_df = pd.DataFrame({
        'text': X_test.values,
        'label': y_test.values
    })


    os.makedirs(output_dir, exist_ok=True)

    # Save processed datasets
    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False
    )

    # Display dataset information
    print("Train Shape:", train_df.shape)
    print("Validation Shape:", val_df.shape)
    print("Test Shape:", test_df.shape)

    print("\nDatasets saved successfully inside:")
    print(output_dir)

    return train_df, val_df, test_df

In [53]:
output_dir = "preprocessed_data/pipeline_1"
pipeline_1(df_imdb, output_dir)

Starting Pipeline 1 Preprocessing...
Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)

Datasets saved successfully inside:
preprocessed_data/pipeline_1


(                                                    text  label
 0      always huge james bond fanatic seen almost fil...      1
 1      christian say movie terrible acting unreal sit...      0
 2      neatly sandwiched stranger small film noir pic...      1
 3      year ago follow soap tv curious movie rewarded...      1
 4      here gritty getthebad guy revenge story starri...      1
 ...                                                  ...    ...
 19995  popular radio storyteller gabriel onerobin wil...      1
 19996  throughout film might think film kid well main...      1
 19997  quite producer appalling adaptation trying imp...      0
 19998  trigger man definitely boring silliest movie i...      0
 19999  movie gem move soft firm resolutionbr br cauti...      1
 
 [20000 rows x 2 columns],
                                                    text  label
 0     one best hong kong action film around tense ex...      1
 1     time best computer animation classic even thou...      1

----------------

# **Pipeline 2: NLP Preprocessing Without Stopword Removal**

This preprocessing pipeline is designed as the second experimental preprocessing
strategy for the IMDB sentiment analysis project.

Unlike Pipeline 1, this version preserves stopwords in order to analyze how
functional words influence sentiment classification performance.

The objective of this pipeline is to evaluate whether maintaining stopwords
improves the representation of contextual and semantic information in movie
reviews, especially for sentiment-related expressions involving negation.

The pipeline performs the following operations sequentially:

1. Remove special characters and numbers
2. Tokenize the text into words
3. Convert all tokens to lowercase
4. Apply lemmatization
5. Reconstruct the cleaned text

This pipeline prioritizes:
- Preservation of contextual information
- Retention of negation words
- Comparative analysis against Pipeline 1
- Compatibility with multiple NLP architectures

Unlike Pipeline 1, stopwords such as:
- not
- no
- never

are preserved because they may contain important sentiment information.

After preprocessing, the dataset is divided into:
- Training set
- Validation set
- Test set

The processed datasets are stored inside a directory called:

```text
preprocessed_data_pipeline_2/
```

This allows the same preprocessing strategy to be reused later for:
- TF-IDF experiments
- Embedding architectures
- Classification models
- Comparative NLP experiments


## **Pipeline 2 Implementation**

In [54]:

def preprocessing_pipeline_2(text):

    """
    Pipeline 2 for IMDB preprocessing.

    Steps:
    1. Remove special characters
    2. Tokenization
    3. Lowercasing
    4. Lemmatization
    5. Join cleaned tokens

    Returns:
        Cleaned text as a string.
    """

    # Remove special characters and numbers
    # Keeps only alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))

    # Tokenization
    tokens = word_tokenize(text)

    # Stores processed words
    processed_tokens = []

    # Token processing loop
    for word in tokens:

        # Lowercase conversion and whitespace cleaning
        word = word.lower().strip()
        # Filtering conditions:
        # - Remove stopwords
        # - Remove short tokens
        # - Keep only alphabetic tokens
        if (
            word not in stop_words
            and len(word) > 1
            and word.isalpha()
        ):

            # Lemmatization
            word = lemmatizer.lemmatize(word)

            # Add cleaned token
            processed_tokens.append(word)

    # Reconstruct cleaned text
    cleaned_text = " ".join(processed_tokens)

    return cleaned_text


# Complete Pipeline Function
def pipeline_2(df, output_dir="preprocessed_data_pipeline_2"):

    """
    Applies preprocessing pipeline 2,
    performs train/validation/test split,
    and saves processed datasets.
    """
    print("Starting Pipeline 2 Preprocessing...")
    df = df.copy()
    df['clean_text'] = df['text'].apply(preprocessing_pipeline_1)
    df_clean = df[['clean_text', 'label']]

    # Train / Validation / Test Split
    # 80% train
    # 20% temporary
    X_train, X_temp, y_train, y_temp = train_test_split(
        df_clean['clean_text'],
        df_clean['label'],
        test_size=0.2,
        random_state=42,
        stratify=df_clean['label']
    )

    # 10% validation
    # 10% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    # Create DataFrames
    train_df = pd.DataFrame({
        'text': X_train.values,
        'label': y_train.values
    })

    val_df = pd.DataFrame({
        'text': X_val.values,
        'label': y_val.values
    })

    test_df = pd.DataFrame({
        'text': X_test.values,
        'label': y_test.values
    })


    os.makedirs(output_dir, exist_ok=True)

    # Save processed datasets
    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False
    )

    # Display dataset information
    print("Train Shape:", train_df.shape)
    print("Validation Shape:", val_df.shape)
    print("Test Shape:", test_df.shape)

    print("\nDatasets saved successfully inside:")
    print(output_dir)

    return train_df, val_df, test_df

In [55]:
output_dir = "preprocessed_data/pipeline_2"
pipeline_2(df_imdb, output_dir)

Starting Pipeline 2 Preprocessing...
Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)

Datasets saved successfully inside:
preprocessed_data/pipeline_2


(                                                    text  label
 0      always huge james bond fanatic seen almost fil...      1
 1      christian say movie terrible acting unreal sit...      0
 2      neatly sandwiched stranger small film noir pic...      1
 3      year ago follow soap tv curious movie rewarded...      1
 4      here gritty getthebad guy revenge story starri...      1
 ...                                                  ...    ...
 19995  popular radio storyteller gabriel onerobin wil...      1
 19996  throughout film might think film kid well main...      1
 19997  quite producer appalling adaptation trying imp...      0
 19998  trigger man definitely boring silliest movie i...      0
 19999  movie gem move soft firm resolutionbr br cauti...      1
 
 [20000 rows x 2 columns],
                                                    text  label
 0     one best hong kong action film around tense ex...      1
 1     time best computer animation classic even thou...      1

# **Pipeline 3: NLP Preprocessing Without Lemmatization**

This preprocessing pipeline is designed as the third experimental preprocessing
strategy for the IMDB sentiment analysis project.

Unlike Pipeline 1, this version removes stopwords but does not apply
lemmatization. The objective is to analyze how preserving the original
morphological structure of words affects text representation and classification
performance.

This pipeline allows comparison between:
- Lemmatized representations
- Non-lemmatized representations
- Vocabulary dimensionality differences
- Effects of word normalization on sentiment analysis

The pipeline performs the following operations sequentially:

1. Remove special characters and numbers
2. Tokenize the text into words
3. Convert all tokens to lowercase
4. Remove stopwords
5. Reconstruct the cleaned text

This pipeline prioritizes:
- Simplicity
- Faster preprocessing
- Preservation of original word forms
- Comparative analysis against lemmatized pipelines

Unlike Pipeline 1:
- no lemmatization is applied
- original grammatical forms are preserved

Examples:

| Original Word | Pipeline 1 | Pipeline 3 |
|---|---|---|
| running | run | running |
| movies | movie | movies |
| actors | actor | actors |

After preprocessing, the dataset is divided into:
- Training set
- Validation set
- Test set

The processed datasets are stored inside a directory called:

```text
preprocessed_data_pipeline_3/
```

This allows the same preprocessing strategy to be reused later for:
- TF-IDF experiments
- Bag of Words representations
- Embedding architectures
- Classification models
- Comparative NLP experiments

## **Pipeline 3 Implementation**

In [56]:
def preprocessing_pipeline_3(text):

    """
    Pipeline 3 for IMDB preprocessing.

    Steps:
    1. Remove special characters
    2. Tokenization
    3. Lowercasing
    4. Stopword removal
    5. Join cleaned tokens

    Returns:
        Cleaned text as a string.
    """

    # Remove special characters and numbers
    # Keeps only alphabetic characters
    text = re.sub(r'[^a-zA-Z\s]', '', str(text))

    # Tokenization
    tokens = word_tokenize(text)

    # Stores processed words
    processed_tokens = []

    # Token processing loop
    for word in tokens:

        # Lowercase conversion and whitespace cleaning
        word = word.lower().strip()
        # Filtering conditions:
        # - Remove stopwords
        # - Remove short tokens
        # - Keep only alphabetic tokens
        if (
            word not in stop_words
            and len(word) > 1
            and word.isalpha()
        ):
            # Add cleaned token
            processed_tokens.append(word)

    # Reconstruct cleaned text
    cleaned_text = " ".join(processed_tokens)

    return cleaned_text


# Complete Pipeline Function
def pipeline_3(df, output_dir="preprocessed_data_pipeline_3"):

    """
    Applies preprocessing pipeline 3,
    performs train/validation/test split,
    and saves processed datasets.
    """
    print("Starting Pipeline 3 Preprocessing...")
    df = df.copy()
    df['clean_text'] = df['text'].apply(preprocessing_pipeline_1)
    df_clean = df[['clean_text', 'label']]

    # Train / Validation / Test Split
    # 80% train
    # 20% temporary
    X_train, X_temp, y_train, y_temp = train_test_split(
        df_clean['clean_text'],
        df_clean['label'],
        test_size=0.2,
        random_state=42,
        stratify=df_clean['label']
    )

    # 10% validation
    # 10% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    # Create DataFrames
    train_df = pd.DataFrame({
        'text': X_train.values,
        'label': y_train.values
    })

    val_df = pd.DataFrame({
        'text': X_val.values,
        'label': y_val.values
    })

    test_df = pd.DataFrame({
        'text': X_test.values,
        'label': y_test.values
    })


    os.makedirs(output_dir, exist_ok=True)

    # Save processed datasets
    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False
    )

    # Display dataset information
    print("Train Shape:", train_df.shape)
    print("Validation Shape:", val_df.shape)
    print("Test Shape:", test_df.shape)

    print("\nDatasets saved successfully inside:")
    print(output_dir)

    return train_df, val_df, test_df

In [57]:
output_dir = "preprocessed_data/pipeline_3"
pipeline_3(df_imdb, output_dir)

Starting Pipeline 3 Preprocessing...
Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)

Datasets saved successfully inside:
preprocessed_data/pipeline_3


(                                                    text  label
 0      always huge james bond fanatic seen almost fil...      1
 1      christian say movie terrible acting unreal sit...      0
 2      neatly sandwiched stranger small film noir pic...      1
 3      year ago follow soap tv curious movie rewarded...      1
 4      here gritty getthebad guy revenge story starri...      1
 ...                                                  ...    ...
 19995  popular radio storyteller gabriel onerobin wil...      1
 19996  throughout film might think film kid well main...      1
 19997  quite producer appalling adaptation trying imp...      0
 19998  trigger man definitely boring silliest movie i...      0
 19999  movie gem move soft firm resolutionbr br cauti...      1
 
 [20000 rows x 2 columns],
                                                    text  label
 0     one best hong kong action film around tense ex...      1
 1     time best computer animation classic even thou...      1

# **Pipeline 0: Raw Dataset Without Preprocessing**

This preprocessing pipeline is designed as the baseline experimental strategy
for the IMDB sentiment analysis project.

Unlike the other preprocessing pipelines, Pipeline 0 does not apply any
text cleaning or transformation operations. The original movie reviews are
preserved exactly as they appear in the dataset.

The objective of this pipeline is to establish a reference baseline for
comparing the impact of preprocessing techniques on NLP model performance.

Pipeline 0 allows direct comparison between:
- Raw textual representations
- Cleaned textual representations
- Preprocessed vocabularies
- Different normalization strategies

This pipeline performs only the following operations:

1. Preserve the original text
2. Divide the dataset into:
   - Training set
   - Validation set
   - Test set
3. Save the resulting datasets

No preprocessing operations are applied:
- No tokenization
- No lowercasing
- No stopword removal
- No lemmatization
- No normalization

This pipeline prioritizes:
- Baseline experimentation
- Preservation of original textual structure
- Comparative evaluation against preprocessing pipelines
- Analysis of preprocessing impact on model performance

After dataset splitting, the processed datasets are stored inside a directory called:

```text
preprocessed_data_pipeline_0/
```


## **Pipeline 0 Implementation**

In [58]:
def pipeline_no_preprocessing(df, output_dir="preprocessed_data_pipeline_3"):
    """
    Applies preprocessing pipeline 0,
    performs train/validation/test split,
    and saves processed datasets.
    """
    print("Starting Pipeline 0 Preprocessing...")
    df = df.copy()

    # Train / Validation / Test Split
    # 80% train
    # 20% temporary
    X_train, X_temp, y_train, y_temp = train_test_split(
        df['text'],
        df['label'],
        test_size=0.2,
        random_state=42,
        stratify=df['label']
    )

    # 10% validation
    # 10% test
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp,
        y_temp,
        test_size=0.5,
        random_state=42,
        stratify=y_temp
    )

    # Create DataFrames
    train_df = pd.DataFrame({
        'text': X_train.values,
        'label': y_train.values
    })

    val_df = pd.DataFrame({
        'text': X_val.values,
        'label': y_val.values
    })

    test_df = pd.DataFrame({
        'text': X_test.values,
        'label': y_test.values
    })


    os.makedirs(output_dir, exist_ok=True)

    # Save processed datasets
    train_df.to_csv(
        os.path.join(output_dir, "train.csv"),
        index=False
    )

    val_df.to_csv(
        os.path.join(output_dir, "validation.csv"),
        index=False
    )

    test_df.to_csv(
        os.path.join(output_dir, "test.csv"),
        index=False
    )

    # Display dataset information
    print("Train Shape:", train_df.shape)
    print("Validation Shape:", val_df.shape)
    print("Test Shape:", test_df.shape)

    print("\nDatasets saved successfully inside:")
    print(output_dir)

    return train_df, val_df, test_df

In [59]:
output_dir = "preprocessed_data/pipeline_0"
pipeline_no_preprocessing(df_imdb, output_dir)

Starting Pipeline 0 Preprocessing...
Train Shape: (20000, 2)
Validation Shape: (2500, 2)
Test Shape: (2500, 2)

Datasets saved successfully inside:
preprocessed_data/pipeline_0


(                                                    text  label
 0      I have always been a huge James Bond fanatic! ...      1
 1      I am a Christian and I say this movie had terr...      0
 2      Neatly sandwiched between THE STRANGER, a smal...      1
 3      Years ago I did follow a soap on TV. So I was ...      1
 4      Here's a gritty, get-the-bad guys revenge stor...      1
 ...                                                  ...    ...
 19995  Popular radio storyteller Gabriel No one(Robin...      1
 19996  Throughout this film, you might think this fil...      1
 19997  Quite what the producers of this appalling ada...      0
 19998  "Trigger Man" is definitely the most boring an...      0
 19999  This movie is a Gem because it moves with soft...      1
 
 [20000 rows x 2 columns],
                                                    text  label
 0     This is one of the best Hong Kong (action) fil...      1
 1     This has to be the all time best computer anim...      1